# An Investigation into the Causes and Mitigation of Barren Plateaus in Variational Quantum Algorithms

**QOSF Monthly Challenge - [Jun 2025]**

**Author:** Tan Jun Liang

---

## Abstract

Variational Quantum Algorithms (VQAs) represent a leading paradigm for leveraging near-term quantum hardware. 
However, their scalability is critically hampered by the "barren plateau" phenomenon, 
where training gradients vanish exponentially with system size. 

This notebook presents a systematic investigation into this issue. 

We first review the primary theoretical causes of barren plateaus—including those induced by ansatz depth, 
cost function choice, and hardware noise—citing seminal works in the field. 

We then empirically demonstrate these phenomena through a series of targeted experiments. 

Finally, we implement and evaluate several mitigation strategies, 
from simple initialization techniques to more advanced, gradient-guided ansatz construction like ADAPT-VQE, providing a comprehensive and practical guide to navigating the training landscapes of quantum neural networks.

In [2]:
# --- General Imports ---
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# --- Qiskit Imports ---
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_algorithms.gradients import ParamShiftEstimatorGradient
from qiskit_algorithms.optimizers import SPSA

# --- Matplotlib settings for prettier plots ---
# Note: some settings may not apply to the widget backend
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 14})

## 1. The Theory of Barren Plateaus: A Review

The practical utility of any VQA hinges on the ability of a classical optimizer to successfully train a Parameterized Quantum Circuit (PQC). The discovery of barren plateaus revealed a fundamental obstacle to this process. Research has identified several distinct mechanisms that can lead to a flat, untrainable optimization landscape.

### 1.1 Depth-Induced Barren Plateaus

The original work by **McClean et al. [1]** identified that deep PQCs with random parameter initializations form approximate "2-designs." A 2-design is a distribution of unitary transformations that mimics the statistical properties of the full space of all possible unitaries (the Haar measure) up to the second moment. This high degree of scrambling effectively randomizes the output, causing the expectation value of any observable to concentrate exponentially around a fixed value (typically zero), leading to vanishing gradients.

### 1.2 Cost Function-Induced Barren Plateaus

A subsequent discovery by **Cerezo et al. [2]** showed that barren plateaus can exist even in shallow circuits if the cost function is sufficiently *global*. A global cost function involves observables that act non-trivially on many qubits (e.g., measuring the parity of all qubits, `ZZ...Z`). Due to the phenomenon of concentration of measure, the expectation values of such observables also concentrate exponentially, leading to vanishing gradients independent of circuit depth. Conversely, **local cost functions**, which measure only a few qubits, are immune to this specific mechanism.

### 1.3 Noise-Induced Barren Plateaus (NIBPs)

Even a VQA with a shallow circuit and a local cost function can fail if subjected to sufficient noise. As shown by **Wang et al. [3]**, the presence of global quantum noise (i.e., noise channels acting on all qubits) can itself induce a barren plateau. The noise effectively contracts the reachable state space towards the maximally mixed state, flattening the landscape and destroying the gradient, a phenomenon termed Noise-Induced Barren Plateaus (NIBPs).

This notebook will empirically investigate all three phenomena.

In [ ]:
# define useful function
def build_pqc(num_qubits, depth):
    """Builds the hardware-efficient PQC."""
    qc = QuantumCircuit(num_qubits)
    params = []
    for d in range(depth):
        for i in range(num_qubits):
            param = Parameter(f'p_{d}_{i}')
            params.append(param)
            qc.ry(param, i)
        # Add a circular entangling layer
        for i in range(num_qubits):
            qc.cz(i, (i + 1) % num_qubits)
    return qc, params

# Instantiate the primitives with our high-performance Aer Estimator
estimator = AerEstimator()
gradient = ParamShiftEstimatorGradient(estimator)

## 2. Experimental Setup

### 2.1 Visualizing the Vanishing Gradient

Before we attempt to train our circuit, let's prove that a barren plateau exists for our chosen setup. To do this, we can run a simple experiment:

1.  **For a given number of qubits**, build our PQC.
2.  **Randomly initialize** the circuit's parameters over their full range (`[0, 2π]`).
3.  **Calculate the gradient** of the cost function with respect to a single parameter.
4.  **Repeat** this process many times with different random initializations and calculate the **variance** of the resulting gradients.

The plot below shows the result of this experiment.

<img src="images/output.png" width="600">

#### Interpreting the Plot:

This plot reveals the core of the barren plateau problem.

*   **What it shows:** The y-axis represents the **gradient variance**, and the x-axis is the **number of qubits**.
*   **The Key Trend:** The most important feature is the steep, downward slope. Because the y-axis is on a **logarithmic scale**, this straight downward line indicates an **exponential decay**. The gradient variance isn't just decreasing—it's vanishing exponentially fast as we add more qubits.
*   **The Implication:** What does this mean for training? A gradient variance approaching zero implies that for almost any random initialization, the gradient itself will be a value extremely close to zero. An optimizer that receives a zero-gradient has no "signal" to guide its search for better parameters. It becomes stuck on a vast, flat 'plateau' in the optimization landscape, unable to learn.

This is the barren plateau phenomenon in action, and it is the primary reason why our baseline training will fail. Now, let's confirm that failure and then learn how to fix it.

*(This plot was pre-computed to save you time. The code to generate it is in the collapsed cell below for your reference, but **you do not need to run it**.)*

In [6]:
# --- WARNING: THIS CELL IS SLOW AND FOR REFERENCE ONLY ---

# qubit_counts = [4, 6, 8, 10]
# variances = []
# n_trials = 50 # Number of random initializations
# 
# for n_qubits in tqdm(qubit_counts):
#     pqc, params = build_pqc(n_qubits, depth=n_qubits)
#     local_observable = SparsePauliOp("Z" + "I" * (n_qubits - 1))
#     grads = []
#     for _ in tqdm(range(n_trials)):
#         rand_params = np.random.uniform(0, 2 * np.pi, len(params))
#         grad_result = gradient.run(pqc, local_observable, [rand_params]).result().gradients[0][0]
#         grads.append(grad_result)
#     variances.append(np.var(grads))
# 
# # --- Plotting the result ---
# plt.plot(qubit_counts, variances, 'o-', label='Gradient Variance')
# plt.yscale('log')
# plt.xlabel('Number of Qubits')
# plt.ylabel('Gradient Variance (log scale)')
# plt.title('Demonstration of the Barren Plateau')
# plt.legend()
# plt.show()

To investigate these phenomena, we will use a consistent experimental setup.

*   **Ansatz:** A hardware-efficient ansatz with layers of `Ry` rotations and `CZ` entangling gates.
*   **Simulator:** The high-performance `qiskit-aer` `Estimator`.
*   **Optimizer:** The gradient-free `SPSA` optimizer, which is robust for noisy, stochastic landscapes. Its use of only two circuit executions per step allows for rapid experimentation.
*   **Task:** Train the PQC to produce a target expectation value of `0.5` for a given observable. The cost function is the Mean Squared Error.

In [7]:
# ==============================================================================
# 1. REDUCE PROBLEM SIZE for faster simulation
# ==============================================================================
NUM_QUBITS = 6
DEPTH = 6
Y_target = 0.5 # The target expectation value

# --- Build the main PQC for the challenge ---
pqc, params = build_pqc(NUM_QUBITS, DEPTH)

# ==============================================================================
# 2. CREATE A COST FUNCTION FOR SPSA (Gradient-Free)
# ==============================================================================
def cost_function_for_spsa(p_values, observable):
    """
    Calculates ONLY the cost. SPSA estimates the gradient internally.
    This requires only ONE call to the estimator per optimizer step.
    """
    est_job = estimator.run(pqc, observable, [p_values])
    exp_val = est_job.result().values[0]
    cost = (exp_val - Y_target)**2
    return cost

# ==============================================================================
# 3. CREATE A DEDICATED TRAINING LOOP FOR SPSA
# ==============================================================================
def run_training_spsa(initial_params, cost_func, optimizer, title="", live_output=False):
    """A training loop designed for SPSA."""
    fig = plt.figure()
    ax = fig.add_subplot(1, 1, 1)
    cost_history = []
    params_current = initial_params

    # The SPSA.minimize function handles the entire optimization loop.
    # We just need to give it a function to minimize and the starting point.
    def objective_function(p):
        cost = cost_func(p)
        if live_output:
            clear_output(wait=False)
            # Store history for plotting
            ax.plot(cost_history, color='blue')
            ax.set_xlabel("Optimizer Steps")
            ax.set_ylabel("Cost (MSE)")
            ax.set_title(title)
            display(fig)
            cost = cost_func(p)
        cost_history.append(cost)
        return cost

    # SPSA will call the objective_function max_iter times.
    result = optimizer.minimize(
        fun=objective_function,
        x0=params_current,
    )

    # Note: SPSA's internal loop gives a noisy cost history.
    # We return the history we captured for a clearer plot.
    return cost_history, result.x

## 3. Experiments in VQA Trainability

### 3.1 Experiment 1: The Baseline Case

We first combine a deep circuit (`depth=qubits`) with a local cost function and random initialization. Based on the theory of **McClean et al. [1]**, we hypothesize that this setup will exhibit a depth-induced barren plateau and fail to train.

In [ ]:
# --- Baseline Training with SPSA ---
print("Running Baseline Training (Random Initialization)...")
# SPSA requires a callback to track progress, which we built into our loop.

def callbackfun(nevals, params, fval, stepsize, acceptedstep):
    global iter
    print(f'Iteration: {iter} Number of evaluations: {nevals}')
    iter += 1

optimizer = SPSA(maxiter=100)
initial_params_baseline = np.random.uniform(0, 2 * np.pi, len(params))

# Use a local observable for the baseline
local_observable = SparsePauliOp("Z" + "I" * (NUM_QUBITS - 1))
cost_func_local = lambda p: cost_function_for_spsa(p, local_observable)

baseline_cost, _ = run_training_spsa(initial_params_baseline, cost_func_local, optimizer,live_output=False)

# --- Plotting ---
plt.plot(baseline_cost, label='Baseline (Random Init)')
plt.xlabel('Optimizer Steps')
plt.ylabel('Cost (MSE)')
plt.title('Baseline Training Attempt with SPSA')
plt.legend()
plt.show()

### 3.2 Experiment 2: Mitigation Strategies

#### Task 1: Tackling Depth-Induced Barren Plateaus

According to **Grant et al. [4]**, initializing parameters near zero keeps the PQC close to the identity, preventing it from forming a 2-design too quickly. We hypothesize this will mitigate the barren plateau from Experiment 1.

In [ ]:
# YOUR CODE HERE
# 1. Define a new set of initial parameters with narrow initialization.
# 2. Run the training_loop with these new parameters.
# 3. Plot the resulting cost history.

print("Running Training with Narrow Initialization...")
# Hint: Change the initialization range
initial_params_narrow = np.random.uniform(0, 0.01, len(params))

# We use the same local cost function and optimizer
narrow_cost, _ = run_training_spsa(initial_params_narrow, cost_func_local, optimizer,live_output=False)

plt.plot(narrow_cost, label='Narrow Init', color='green')
plt.xlabel('Optimizer Steps')
plt.ylabel('Cost (MSE)')
plt.title('Training with Narrow Parameter Initialization')
plt.legend()
plt.show()

#### Task 2: Inducing a Cost-Function Barren Plateau

We now test the theory of **Cerezo et al. [2]**. We will use the successful narrow initialization from Task 1 but switch to a global cost function. We hypothesize that this will re-introduce a barren plateau, causing training to fail again.

In [ ]:
# YOUR CODE HERE
# Generate the final comparison plot showing all loss histories on one graph.
plt.plot(baseline_cost, label='Baseline (Local Cost, Random Init)')
plt.plot(narrow_cost, label='Mitigated (Local Cost, Narrow Init)', color='green')
plt.plot(global_cost, label='Failed (Global Cost, Narrow Init)', color='red')

plt.xlabel('Optimizer Steps')
plt.ylabel('Cost (MSE)')
plt.title('Comparison of Training Strategies')
plt.legend(loc='best')
plt.show()

### 3.3 Experiment 3 (Advanced): Investigating Noise-Induced Barren Plateaus

Finally, we investigate the NIBP phenomenon described by **Wang et al. [3]**. We will return to our "solved" setup from Task 1 (narrow init, local cost) but add a global depolarizing noise channel to our simulator.

**Hypothesis:** The presence of global noise will induce a barren plateau, causing our previously successful training to fail.

**Challenge:** Implement a simple noise model and show that the training fails.

In [ ]:
# --- Import tools for creating a noise model ---
from qiskit_aer.noise import NoiseModel, depolarizing_error

# 1. Create a simple noise model with global depolarizing noise
noise_model = NoiseModel()

# Add a 1% depolarizing error after every CNOT gate
error = depolarizing_error(0.01, 2)
noise_model.add_all_qubit_quantum_error(error, ['cz'])
print("Noise Model created:", noise_model)

# 2. Create a new Estimator that uses this noise model
noisy_estimator = AerEstimator(backend_options={"noise_model": noise_model})
# 3. Define a new cost function that uses the noisy estimator
def noisy_cost_function(p_values, observable):
    """Calculates cost using the noisy estimator."""
    est_job = noisy_estimator.run(pqc, observable, [p_values])
    exp_val = est_job.result().values[0]
    cost = (exp_val - Y_target)**2
    return cost
# 4. Run the training
print("\nRunning Training with Noise (Narrow Init, Local Cost)...")
optimizer_spsa = SPSA(maxiter=100)
initial_params_narrow = np.random.uniform(0, 0.01, len(params))
cost_func_noisy = lambda p: noisy_cost_function(p, local_observable)
# We use the standard training loop for this
def run_training_standard(initial_params, cost_func, optimizer):
    cost_history = []
    def objective(p):
        cost = cost_func(p)
        cost_history.append(cost)
        return cost
    res = optimizer.minimize(fun=objective, x0=initial_params)
    return cost_history, res.x
nibp_cost, _ = run_training_standard(initial_params_narrow, cost_func_noisy, optimizer_spsa)
# --- Plotting ---
plt.figure()
plt.plot(nibp_cost, label='With Noise (NIBP)', color='orange')
plt.xlabel('Optimizer Steps')
plt.ylabel('Cost (MSE)')
plt.title('Training Failure due to Noise-Induced Barren Plateau')
plt.legend()
plt.show()

Noise Model created: NoiseModel:
  Basis gates: ['cx', 'cz', 'id', 'rz', 'sx']
  Instructions with noise: ['cz']
  All-qubits errors: ['cz']

Running Training with Noise (Narrow Init, Local Cost)...


## 4. Bonus Challenge: Algorithmic Mitigation with ADAPT-VQE

A highly effective, state-of-the-art mitigation strategy is to not use a fixed ansatz at all, but to build one iteratively. The **ADAPT-VQE** algorithm, introduced by **Grimsley et al. [5]**, does exactly this.

**Algorithm:**
1.  Define a "pool" of operators (e.g., all possible Pauli strings).
2.  Start with a simple reference state (e.g., Hartree-Fock, or in our case, the `|0...0>` state).
3.  **Loop:**
    a. Calculate the gradient of the cost function with respect to each operator in the pool.
    b. Select the operator with the largest gradient magnitude. This is the direction of steepest descent.
    c. Add this operator (as an exponential `exp(-iθP)`) to the end of the current ansatz.
    d. Optimize all parameters in the newly extended ansatz.
4.  Repeat until the largest gradient in the pool is below a threshold.

This method builds a compact, problem-tailored ansatz that is highly resistant to barren plateaus by design. Implementing this is a significant but rewarding challenge.

## 5. Conclusion

This investigation empirically confirmed the primary theoretical causes of barren plateaus in VQAs. We demonstrated that deep, randomly initialized circuits and the use of global cost functions are significant obstacles to trainability. Furthermore, we showed that even an otherwise well-behaved VQA can be rendered untrainable by the presence of global hardware noise, confirming the existence of NIBPs.

Simple mitigation strategies, such as narrow parameter initialization and the use of local observables, proved effective against their corresponding barren plateau mechanisms. For a robust VQA, it is clear that a holistic approach, considering all potential pitfalls from ansatz design to cost function definition and noise resilience, is paramount. More advanced, adaptive methods like ADAPT-VQE offer a promising path forward for constructing scalable and trainable models.

### Write summary

In the markdown cell below, please describe your findings.
*   Which mitigation strategy performed the best and why?
*   Why did the global cost function fail to train, even with good initialization?
*   What are the trade-offs for each method (e.g., implementation complexity)?
*   Did you try the bonus challenge or any other creative ideas? If so, what were they and how did they perform?

---

*... Your analysis here ...*


| Strategy                      | Implementation Complexity(Trivial/High) | Performance(Failed/Success) | Key Takeaway                                                                |
| ----------------------------- | ------------------------- | ----------- | --------------------------------------------------------------------------- |
| Random Init / Local Cost      | (write here) | (write here)  |  (write here) |
| Narrow Init / Local Cost      | (write here) | (write here)  | (write here)  |
| Narrow Init / Global Cost     | (write here) | (write here)  | (write here)  |
| Layer-by-Layer Training (Bonus) | (write here) | (write here) | (write here) |

## 6. References

[1] McClean, J. R., et al. "Barren plateaus in quantum neural network training landscapes." *Nature Communications* 9.1 (2018): 4812.

[2] Cerezo, M., et al. "Cost function dependent barren plateaus in shallow parametrized quantum circuits." *Nature Communications* 12.1 (2021): 1791.

[3] Wang, S., et al. "Noise-induced barren plateaus in variational quantum algorithms." *Nature Communications* 12.1 (2021): 6996.

[4] Grant, E., et al. "An initialization strategy for addressing barren plateaus in parametrized quantum circuits." *Quantum* 3 (2019): 214.

[5] Grimsley, H. R., et al. "An adaptive variational algorithm for exact molecular simulations on a quantum computer." *Nature Communications* 10.1 (2019): 3007.